# Proyecto LSP — Exportar el modelo a TensorFlow.js (para la demo web)

Convierte `modelo_pucp305_final.keras` → formato **TensorFlow.js** (`model.json` + pesos)
para que la demo HTML (`web/lsp_demo.html`) corra el modelo en el navegador, sin Python.

Genera un `web_model.zip` que descargas y descomprimes en `web/web_model/` (junto al HTML).

> **Nota:** el BiLSTM convierte bien. La celda de conversión incluye un **parche
> Keras 3 → TF.js** del `model.json` (renombra `batch_shape`→`batch_input_shape` y
> normaliza `dtype`), sin el cual el navegador falla con *"An InputLayer should be
> passed either a batchInputShape or an inputShape"*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# tensorflowjs instala sus propias deps, INCLUIDA jax (su converter la importa).
# OJO: a diferencia de los notebooks de inferencia (04/05/06-setup), aquí NO se
# desinstala jax — hacerlo rompe `import tensorflowjs` (ModuleNotFoundError: jax).
!pip install -q tensorflowjs

In [ ]:
import shutil, json
from pathlib import Path
import tensorflow as tf
import tensorflowjs as tfjs

M = Path('/content/drive/MyDrive/PUCP305_models')
OUT = Path('/content/web_model'); OUT.mkdir(exist_ok=True)

model = tf.keras.models.load_model(M / 'modelo_pucp305_final.keras', compile=False)
print('Modelo cargado:', model.input_shape, '->', model.output_shape)

# Keras -> TensorFlow.js (LayersModel, lo carga el HTML con tf.loadLayersModel)
tfjs.converters.save_keras_model(model, str(OUT))
print('Conversion OK.')

# --- Parche Keras 3 -> TF.js ---
# Keras 3 escribe el model.json con claves que TF.js (formato Keras 2) no entiende:
#   * InputLayer usa 'batch_shape'  -> TF.js espera 'batch_input_shape'
#   * 'dtype' viene como objeto DTypePolicy -> TF.js espera el string "float32"
# Sin esto, el navegador falla con "An InputLayer should be passed either a
# batchInputShape or an inputShape".
def _parche_keras3(o):
    if isinstance(o, dict):
        if 'batch_shape' in o and 'batch_input_shape' not in o:
            o['batch_input_shape'] = o.pop('batch_shape')
        d = o.get('dtype')
        if isinstance(d, dict):
            name = d.get('config', {}).get('name') or d.get('class_name')
            o['dtype'] = name if isinstance(name, str) else 'float32'
        for v in o.values():
            _parche_keras3(v)
    elif isinstance(o, list):
        for v in o:
            _parche_keras3(v)

mj = OUT / 'model.json'
data = json.load(open(mj, encoding='utf-8'))
_parche_keras3(data['modelTopology'])
json.dump(data, open(mj, 'w', encoding='utf-8'))
print('Parche Keras3->TF.js aplicado a model.json.')

# clases junto al modelo (las usa el HTML)
shutil.copy(M / 'classes_pucp305.json', OUT / 'classes_pucp305.json')
print('\nArchivos generados:')
for p in sorted(OUT.iterdir()):
    print(f'  {p.name:30} {p.stat().st_size/1024:8.1f} KB')

In [ ]:
# Empaquetar y descargar
shutil.make_archive('/content/web_model', 'zip', OUT)
from google.colab import files
files.download('/content/web_model.zip')
print('Descomprime web_model.zip dentro de la carpeta web/ del repo (junto a lsp_demo.html).')